# Line Model — Baseline

In [34]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from torch.optim import AdamW

from modules.plotting import MetricLog, plot_metrics
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.datasets.loading import load_files
from modules.datasets.line_dataset import *
from modules.models.L_baseline_model import *
from modules.tokenizers.base_tokenizer import *

import warnings
warnings.filterwarnings("ignore")

# WORKDIR = r'C:\Programing\code_autocomplete'
WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
LINE_MODEL_NAME = 'line_model_baseline'
print(f"WORKDIR: {WORKDIR}")
print(f"MODEL)NAME: {LINE_MODEL_NAME}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

AttributeError: 'bool' object has no attribute 'any'

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
MODEL)NAME: line_model_baseline
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Training loop

In [35]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(
    model:      LineModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt   = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit  = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])
    PAD = SPECIAL["<PAD>"]

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_steps = 0
        gn = 0.0
        for src, tgt in tqdm(train_dl, desc=f"[Line] Epoch {ep}/{epochs} train",
                             leave=False, unit="batch"):
            src, tgt = src.to(device), tgt.to(device)
            pad_mask = (src == SPECIAL["<PAD>"])
            dec_in   = tgt[:, :-1]
            dec_out  = tgt[:, 1:]
            logits   = model(src, dec_in, src_key_padding_mask=pad_mask)
            loss     = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss  += loss.item()
            t_steps += 1
        tl = t_loss / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_acc = v_steps = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                 leave=False, unit="batch"):
                src, tgt = src.to(device), tgt.to(device)
                pad_mask = (src == PAD)
                dec_in   = tgt[:, :-1]; dec_out = tgt[:, 1:]
                logits   = model(src, dec_in, src_key_padding_mask=pad_mask)
                loss     = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                preds = logits.argmax(-1)
                mask  = dec_out != PAD
                print(preds[mask], dec_out[mask])
                if mask.any():
                    v_acc += (preds[mask] == dec_out[mask]).float().mean().item()
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        va = v_acc  / v_steps if v_steps else 0.0
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=va, grad_norm=gn)
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  val_acc={va:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [36]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                    plot_dir: str = F"{WORKDIR}/plots/{LINE_MODEL_NAME}", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments()
    args = Arguments(epochs=2, max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

   
    tok = CodeTokenizer(vocab_size=8000)
    tok.build(["def foo():\n    return 1\n", "import torch\n"])
   
    # test
    test = "x = foo(1, 2)"
    ids = tok.encode(test)
    print("tokens:", [tok.id2token[i] for i in ids])
    print("UNK count:", sum(1 for i in ids if i == SPECIAL["<UNK>"]))
    # should print zero UNKs


    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    # torch.serialization.add_safe_globals([ModelCfg])

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        lm = LineModel(cfg).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME + "_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(None, lm, None, tokenizer, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── LINE MODEL ──────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        tr_line_ds = LineDataset(tr_txt, tokenizer)
        va_line_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id) 
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters")

        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log = MetricLog()
        print("  Training LINE model")
        train_line_model(line_model, tr_line_dl, va_line_dl, args.epochs, args.lr,
                         device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(None, line_model, None, tokenizer, device)


main()

[Tokenizer] vocab_size=72
tokens: ['<BOS>', 'x', ' ', '\n', '<UNK>', ' ', '\n', 'f', 'o', 'o', '<UNK>', '1', ',', 'f', 'o', 'o', '<UNK>', '1', ' ', '\n', '2', '<UNK>', '<EOS>']
UNK count: 4
[Tokenizer] building from data …
[Data] loaded 100 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Tokenizer] vocab_size=4
[Loading] Started loading
[Data] loaded 100 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model
[LineDataset] 7550 samples
[LineDataset] 862 samples
[Line  Model] 7.38M parameters
  Training LINE model
[Line] DataLoader — 236 train batches, 27 val batches
Epoch 1


[Line] Epoch 1/2 val  :  11%|█         | 3/27 [00:00<00:01, 20.67batch/s]   

tensor([1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1,
        3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1,

[Line] Epoch 1/2 val  :  22%|██▏       | 6/27 [00:00<00:01, 20.60batch/s]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3,
        3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3,
        3, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1,
        1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

...

[Line] Epoch 1/2 val  :  81%|████████▏ | 22/27 [00:01<00:00, 17.85batch/s]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1,

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1,

[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_baseline/line_model_baseline_ep01.png
Epoch 2


KeyboardInterrupt: 